# Notebook 04 — RoPE y ALiBi: codificaciones posicionales modernas

**Bootcamp Bio-LLMs · Módulo 1 · Sesión 4 (extensión opcional)**
Proyecto posdoctoral CICESE — Modelos de lenguaje para venómica integrativa de *Conus*.

---

## Por qué este notebook

Es la **resolución completa del Ejercicio 7.3 del Notebook 02**.

Los modelos Bio-LLM modernos que usarás directamente en el proyecto (Módulos 3-5) **no usan codificación posicional sinusoidal**:

| Modelo | Codificación posicional | Por qué importa para tu proyecto |
|---|---|---|
| **DNABERT** (Ji 2021) | Embeddings aprendidos | Límite duro 512 tokens — no escala |
| **ESM-2** (Lin 2023) | **RoPE** | pLM principal para conotoxinas; extrapola |
| **Nucleotide Transformer v2** (Dalla-Torre 2023) | **RoPE** | gLM principal multispecie; extrapola |
| **DNABERT-2** (Zhou 2024) | **ALiBi** | Maneja secuencias largas sin reentrenamiento |
| **HyenaDNA** (Nguyen 2023) | Sin atención, SSM | Contexto 1 M nt — alternativa total |

Si vas a hacer fine-tuning sobre estos modelos en el Módulo 4, **necesitas entender qué hace RoPE y ALiBi por debajo del capó**.

## Objetivos

1. Entender por qué la codificación sinusoidal del Notebook 02 **falla** al extrapolar más allá del rango de entrenamiento.
2. Implementar **Rotary Position Embeddings (RoPE)** desde cero. Verificar la propiedad de invarianza relativa.
3. Implementar **Attention with Linear Biases (ALiBi)** desde cero. Comparar pendientes por cabeza.
4. **Experimento empírico**: entrenar tres encoders idénticos (sinusoidal vs RoPE vs ALiBi) sobre secuencias de longitud 64 y evaluar su perplejidad en secuencias de longitud 64, 128 y 256.
5. Conectar con la literatura: ¿por qué ESM-2 y NT-v2 eligieron RoPE? ¿Por qué DNABERT-2 eligió ALiBi?

## Pre-requisitos

* Notebooks 01, 02 y 03 completados.
* PyTorch >= 2.0, GPU recomendada para el experimento de la sección 5.

## 0. Imports y configuración

In [ ]:
import math
import random
from typing import Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

## 1. El problema: por qué los embeddings posicionales fallan al extrapolar

Recuerda del Notebook 02 que la codificación posicional sinusoidal **se suma** al embedding del token:

$$x'_t = \text{Embed}(token_t) + PE_t$$

Y luego, dentro de la atención, los scores entre el token en posición $m$ y el token en posición $n$ son:

$$\text{score}(m, n) \propto (x_m + PE_m)^\top W_q^\top W_k (x_n + PE_n)$$

Expandiendo:

$$= \underbrace{x_m^\top W_q^\top W_k x_n}_{\text{contenido}} + \underbrace{x_m^\top W_q^\top W_k PE_n + PE_m^\top W_q^\top W_k x_n}_{\text{cruzados}} + \underbrace{PE_m^\top W_q^\top W_k PE_n}_{\text{posición-posición}}$$

**El problema**: los términos cruzados mezclan contenido con posición de forma no separable. Durante el entrenamiento sólo ves posiciones 0-N. Cuando aparece la posición N+50 en inferencia, los pesos $W_q W_k$ aprendidos producen valores fuera de su distribución de entrenamiento -> la atención colapsa.

**El insight de Su et al. (2021) - RoPE**: en lugar de *sumar* la posición al contenido, **rota** los vectores Q y K por un ángulo que depende de la posición. El producto interno preservará automáticamente sólo la diferencia relativa de posiciones.

**El insight de Press et al. (2021) - ALiBi**: no codifiques la posición en los vectores en absoluto. Añade un *sesgo lineal* a los scores de atención que penaliza distancias largas. Sin parámetros nuevos, extrapolación gratis.

## 2. Rotary Position Embeddings (RoPE)

### 2.1 Formulación matemática

Dado un vector $\mathbf{x} \in \mathbb{R}^d$ en posición $m$, lo dividimos en $d/2$ pares y rotamos cada par por un ángulo $m\theta_i$:

$$\mathbf{x}^{(m)}_{2i:2i+2} = R_{m\theta_i} \cdot \mathbf{x}_{2i:2i+2} = \begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix} \begin{pmatrix} x_{2i} \\ x_{2i+1} \end{pmatrix}$$

donde $\theta_i = 10000^{-2i/d}$ (las mismas frecuencias que el PE sinusoidal).

### 2.2 La propiedad mágica

Si $\tilde{\mathbf{q}}_m = R_m \mathbf{q}$ y $\tilde{\mathbf{k}}_n = R_n \mathbf{k}$, entonces:

$$\langle \tilde{\mathbf{q}}_m, \tilde{\mathbf{k}}_n \rangle = \mathbf{q}^\top R_m^\top R_n \mathbf{k} = \mathbf{q}^\top R_{n-m} \mathbf{k}$$

(porque las rotaciones cumplen $R_m^\top R_n = R_{n-m}$).

-> **El score de atención depende sólo de la diferencia $n-m$**, no de las posiciones absolutas.

### 2.3 Aplicación: sólo a Q y K, no a V

A diferencia de PE sinusoidal (que se suma a los embeddings antes de calcular Q, K, V), RoPE se aplica **dentro** de la atención, sobre Q y K *después* de la proyección, y nunca sobre V.

In [ ]:
def precompute_rope_cache(seq_len, head_dim, base=10000.0, device="cpu"):
    """
    Precomputa cos y sin de los angulos m*theta_i para todas las posiciones m
    y dimensiones i. Convencion LLaMA/ESM-2: split-half.

    Returns
    -------
    cos, sin : tensores (seq_len, head_dim) con los angulos m*theta_i.
    """
    assert head_dim % 2 == 0, "head_dim debe ser par para RoPE"
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    positions = torch.arange(seq_len, device=device).float()
    angles = torch.outer(positions, inv_freq)  # (seq_len, head_dim/2)
    # Duplicar para tener (seq_len, head_dim)
    angles = torch.cat([angles, angles], dim=-1)
    return angles.cos(), angles.sin()


def rotate_half(x):
    """[x1, x2] -> [-x2, x1] donde x1, x2 son las dos mitades de la ultima dimension."""
    d = x.shape[-1]
    x1, x2 = x[..., : d // 2], x[..., d // 2 :]
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    """
    Aplica RoPE a un tensor (B, num_heads, seq_len, head_dim).
    cos, sin : (seq_len, head_dim) -- se broadcastean a (1, 1, seq_len, head_dim).
    """
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return x * cos + rotate_half(x) * sin


# --- Verificacion de la propiedad magica ---
seq_len, head_dim = 8, 32
cos, sin = precompute_rope_cache(seq_len, head_dim)
print(f"cos shape: {tuple(cos.shape)}, sin shape: {tuple(sin.shape)}")

torch.manual_seed(0)
q = torch.randn(1, 1, seq_len, head_dim)
k = q.clone()

q_rot = apply_rope(q, cos, sin)
k_rot = apply_rope(k, cos, sin)

scores_plain = (q[0, 0] @ k[0, 0].T)
scores_rope = (q_rot[0, 0] @ k_rot[0, 0].T)

print("\nScores SIN RoPE (todos iguales porque q=k):")
print(scores_plain.round(decimals=2))
print("\nScores CON RoPE (deberian depender solo de |n-m|):")
print(scores_rope.round(decimals=2))

In [ ]:
# Visualizar la matriz de scores
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].imshow(scores_plain.numpy(), cmap="coolwarm")
axes[0].set_title("Scores SIN RoPE\n(q=k => todos iguales, no informativo)")
axes[0].set_xlabel("n"); axes[0].set_ylabel("m")

axes[1].imshow(scores_rope.numpy(), cmap="coolwarm")
axes[1].set_title("Scores CON RoPE\n(diagonales = constantes -> invarianza relativa)")
axes[1].set_xlabel("n"); axes[1].set_ylabel("m")

plt.tight_layout()
plt.show()

print("\nVerificacion: score(m, m+k) es invariante respecto a m?")
print(f"{'k':<5}{'score(0,k)':<14}{'score(1,k+1)':<14}{'score(2,k+2)':<14}{'?'}")
for k_offset in range(1, 5):
    vals = [scores_rope[m, m + k_offset].item() for m in range(0, 4)]
    is_const = max(vals) - min(vals) < 0.01
    flag = "OK" if is_const else "X"
    print(f"{k_offset:<5}{vals[0]:<14.4f}{vals[1]:<14.4f}{vals[2]:<14.4f}{flag}")

## 3. Attention with Linear Biases (ALiBi)

### 3.1 La idea — radicalmente simple

Press et al. (2021) hacen una pregunta provocadora: y si **eliminamos por completo la codificación posicional** y en su lugar **sesgamos los scores de atención** con una penalización lineal por distancia?

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + m \cdot B\right) V$$

donde $B_{i,j} = -|i - j|$ (versión bidireccional usada en DNABERT-2) o $B_{i,j} = -(i-j)$ con $i \geq j$ (versión causal del paper original).

$m$ es una **pendiente específica por cabeza**, fija (no aprendida). Para $H$ cabezas (potencia de 2):

$$m_h = 2^{-8h/H}, \quad h = 1, 2, \ldots, H$$

Esto da pendientes geométricamente decrecientes: cabezas con $m$ grande ven sólo contexto local, cabezas con $m$ pequeño ven contexto global.

### 3.2 Por qué funciona

* **Sin parámetros nuevos**: la matriz B y las pendientes m son constantes determinísticas.
* **Extrapolación gratuita**: si entrenaste con N=128, puedes evaluar con N=1024 sin nada que ajustar.
* **Sesgo inductivo razonable**: tokens cercanos suelen ser más relevantes — el modelo no tiene que aprenderlo.

In [ ]:
def get_alibi_slopes(num_heads):
    """
    Pendientes ALiBi por cabeza. Para potencias de 2: m_h = 2^(-8*h/H) para h en 1..H.
    """
    def power_of_two_slopes(n):
        start = 2 ** (-2 ** -(math.log2(n) - 3))
        ratio = start
        return [start * (ratio ** i) for i in range(n)]

    if math.log2(num_heads).is_integer():
        return torch.tensor(power_of_two_slopes(num_heads))
    else:
        closest_power = 2 ** math.floor(math.log2(num_heads))
        slopes = power_of_two_slopes(closest_power)
        extra = power_of_two_slopes(2 * closest_power)
        slopes += extra[0::2][: num_heads - closest_power]
        return torch.tensor(slopes)


def build_alibi_bias(seq_len, num_heads, device="cpu", bidirectional=True):
    """
    Construye el sesgo ALiBi de forma (1, num_heads, seq_len, seq_len).
    bidirectional=True (como DNABERT-2): bias = -|i - j| * m_h
    """
    slopes = get_alibi_slopes(num_heads).to(device)  # (H,)
    positions = torch.arange(seq_len, device=device)
    if bidirectional:
        relative = -(positions[:, None] - positions[None, :]).abs().float()
    else:
        relative = positions[None, :] - positions[:, None]
        relative = relative.float().masked_fill(relative > 0, float("-inf"))
    bias = slopes.view(-1, 1, 1) * relative.unsqueeze(0)  # (H, N, N)
    return bias.unsqueeze(0)  # (1, H, N, N)


# --- Inspeccionar las pendientes ---
slopes_8 = get_alibi_slopes(8)
print("Pendientes ALiBi para 8 cabezas:")
for i, s in enumerate(slopes_8):
    print(f"  cabeza {i+1}: m = {s.item():.4f}  (= 2^{math.log2(s.item()):.1f})")

# Visualizar
N, H = 16, 8
bias = build_alibi_bias(seq_len=N, num_heads=H, bidirectional=True)
print(f"\nForma del bias: {tuple(bias.shape)}")

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for h in range(8):
    ax = axes[h // 4, h % 4]
    im = ax.imshow(bias[0, h].numpy(), cmap="coolwarm_r", vmin=-15, vmax=0)
    ax.set_title(f"Cabeza {h+1}  (m = {slopes_8[h]:.4f})")
    ax.set_xlabel("key"); ax.set_ylabel("query")
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle("Sesgo ALiBi bidireccional por cabeza\n(cabezas con m grande -> atencion local)",
             y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

## 4. Encoder unificado con tres modos de codificación posicional

Integramos los tres esquemas en una sola clase `FlexibleEncoderLayer` para hacer comparación apple-to-apple en la sección 5.

In [ ]:
class FlexibleMHA(nn.Module):
    """Multi-head attention que soporta sinusoidal (sin nada extra), RoPE, o ALiBi."""

    def __init__(self, d_model, num_heads, pos_mode="sinusoidal",
                 max_len=512, dropout=0.0):
        super().__init__()
        assert d_model % num_heads == 0
        assert pos_mode in {"sinusoidal", "rope", "alibi"}

        self.d_model, self.num_heads = d_model, num_heads
        self.d_k = d_model // num_heads
        self.pos_mode = pos_mode

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

        if pos_mode == "rope":
            cos, sin = precompute_rope_cache(max_len, self.d_k)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)
        elif pos_mode == "alibi":
            bias = build_alibi_bias(max_len, num_heads, bidirectional=True)
            self.register_buffer("alibi_bias", bias, persistent=False)

    def forward(self, x, mask=None):
        B, N, _ = x.shape
        Q = self.W_q(x).view(B, N, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, N, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, N, self.num_heads, self.d_k).transpose(1, 2)

        # Aplicar RoPE a Q y K
        if self.pos_mode == "rope":
            cos = self.rope_cos[:N]
            sin = self.rope_sin[:N]
            Q = apply_rope(Q, cos, sin)
            K = apply_rope(K, cos, sin)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # Anadir sesgo ALiBi
        if self.pos_mode == "alibi":
            scores = scores + self.alibi_bias[:, :, :N, :N]

        if mask is not None:
            scores = scores.masked_fill(mask, float("-inf"))

        attn = F.softmax(scores, dim=-1)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, N, self.d_model)
        return self.dropout(self.W_o(out))


class FlexibleEncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, pos_mode="sinusoidal",
                 max_len=512, dropout=0.1):
        super().__init__()
        self.mha = FlexibleMHA(d_model, num_heads, pos_mode=pos_mode,
                               max_len=max_len, dropout=dropout)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        x = x + self.dropout(self.mha(self.norm1(x), mask=mask))
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x


class FlexibleMLM(nn.Module):
    """
    Encoder MLM unificado.
    pos_mode='sinusoidal' -> suma PE aditivo al embedding (como Notebook 02).
    pos_mode='rope' o 'alibi' -> NO suma nada al embedding; la posicion entra en la atencion.
    """
    def __init__(self, vocab_size, d_model=128, num_heads=4, num_layers=2,
                 d_ff=512, max_len=512, dropout=0.1, pad_id=0,
                 pos_mode="sinusoidal"):
        super().__init__()
        self.pad_id = pad_id
        self.d_model = d_model
        self.pos_mode = pos_mode

        self.token_embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)

        if pos_mode == "sinusoidal":
            pe = torch.zeros(max_len, d_model)
            position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
            div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
            pe[:, 0::2] = torch.sin(position * div_term)
            pe[:, 1::2] = torch.cos(position * div_term)
            self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

        self.embed_dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList([
            FlexibleEncoderLayer(d_model, num_heads, d_ff, pos_mode=pos_mode,
                                 max_len=max_len, dropout=dropout)
            for _ in range(num_layers)
        ])
        self.final_norm = nn.LayerNorm(d_model)
        self.mlm_head = nn.Linear(d_model, vocab_size, bias=True)
        self.mlm_head.weight = self.token_embedding.weight  # weight tying

    def forward(self, token_ids):
        mask = (token_ids == self.pad_id).unsqueeze(1).unsqueeze(2)
        x = self.token_embedding(token_ids) * math.sqrt(self.d_model)
        if self.pos_mode == "sinusoidal":
            x = x + self.pe[:, : token_ids.size(1)]
        x = self.embed_dropout(x)
        for layer in self.layers:
            x = layer(x, mask=mask)
        x = self.final_norm(x)
        return self.mlm_head(x)


# --- Sanity check ---
for mode in ["sinusoidal", "rope", "alibi"]:
    m = FlexibleMLM(vocab_size=69, d_model=64, num_heads=4, num_layers=2,
                    d_ff=128, max_len=256, pos_mode=mode).to(device)
    out = m(torch.randint(5, 69, (2, 32), device=device))
    n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"mode={mode:<11} | out shape {tuple(out.shape)} | params {n_params:,}")

## 5. Experimento empírico — quién extrapola mejor?

### Diseño

* **Tarea**: MLM sobre codones, corpus sintético de precursores conotoxínicos.
* **Entrenamiento**: secuencias de longitud **64 codones** (≈ 192 nt).
* **Evaluación**: perplejidad MLM en secuencias de longitud {64, 128, 256} — es decir, 1×, 2× y 4× la longitud de entrenamiento.
* **Tres modelos idénticos** en arquitectura, entrenados con la misma semilla y datos. La única diferencia es `pos_mode`.

### Hipótesis (basada en Press et al. 2021, sec. 4)

1. Sinusoidal: degrada notablemente en longitudes > 64 porque los pesos sólo aprendieron a operar sobre PE en ese rango.
2. RoPE: degrada poco — la propiedad relativa se mantiene.
3. ALiBi: extrapola mejor que RoPE, especialmente en longitudes largas.

In [ ]:
# Generador minimo de CDS sinteticas (idea del Notebook 03)
def synth_cds(min_len_codons=64, max_len_codons=320):
    n = random.randint(min_len_codons, max_len_codons)
    codons = []
    bases = "ACGT"
    for _ in range(n):
        r = random.random()
        if r < 0.25:
            codons.append(random.choice(["TGT", "TGC"]))  # cisteinas
        elif r < 0.40:
            codons.append(random.choice(["GGT", "GGC", "GGA"]))  # glicinas
        else:
            codons.append("".join(random.choice(bases) for _ in range(3)))
    return "".join(codons)


class CodonTokenizer:
    SPECIAL = ["[PAD]", "[MASK]", "[CLS]", "[SEP]", "[UNK]"]
    def __init__(self):
        bases = "ACGT"
        codons = [a+b+c for a in bases for b in bases for c in bases]
        self.vocab = self.SPECIAL + codons
        self.t2i = {t:i for i,t in enumerate(self.vocab)}
        self.i2t = {i:t for t,i in self.t2i.items()}
        self.pad_id, self.mask_id = self.t2i["[PAD]"], self.t2i["[MASK]"]
        self.cls_id, self.sep_id = self.t2i["[CLS]"], self.t2i["[SEP]"]
        self.unk_id = self.t2i["[UNK]"]
        self.codon_ids = list(range(len(self.SPECIAL), len(self.vocab)))
    @property
    def vocab_size(self): return len(self.vocab)
    def encode(self, seq, max_len):
        n = len(seq)//3
        ids = [self.t2i.get(seq[3*i:3*i+3].upper(), self.unk_id) for i in range(n)]
        ids = [self.cls_id] + ids + [self.sep_id]
        ids = ids[:max_len]
        return ids + [self.pad_id]*(max_len - len(ids))


def mlm_mask(ids_tensor, tokenizer, p=0.15):
    inputs = ids_tensor.clone()
    labels = ids_tensor.clone()
    special = torch.tensor([tokenizer.pad_id, tokenizer.cls_id, tokenizer.sep_id])
    candidate = ~torch.isin(inputs, special)
    selected = (torch.rand_like(inputs.float()) < p) & candidate
    labels[~selected] = -100
    decision = torch.rand_like(inputs.float())
    apply_mask = selected & (decision < 0.8)
    apply_rand = selected & (decision >= 0.8) & (decision < 0.9)
    inputs[apply_mask] = tokenizer.mask_id
    rand_tokens = torch.tensor(np.random.choice(tokenizer.codon_ids, size=inputs.shape))
    inputs[apply_rand] = rand_tokens[apply_rand]
    return inputs, labels


tok = CodonTokenizer()
print(f"Vocab size: {tok.vocab_size}")

In [ ]:
# Generar corpus
random.seed(SEED)
N_TRAIN, N_VAL = 4000, 600
TRAIN_MAX_LEN = 64

print(f"Generando {N_TRAIN} secuencias de entrenamiento (max_len={TRAIN_MAX_LEN} codones)...")
train_seqs = [synth_cds(min_len_codons=50, max_len_codons=64) for _ in range(N_TRAIN)]
train_token_ids = torch.tensor([tok.encode(s, TRAIN_MAX_LEN) for s in train_seqs])
print(f"Train tensor: {tuple(train_token_ids.shape)}")

# Tres corpus de validacion de DIFERENTES longitudes
val_corpora = {}
for max_len in [64, 128, 256]:
    print(f"Generando {N_VAL} secuencias de validacion (max_len={max_len})...")
    seqs = [synth_cds(min_len_codons=max_len-16, max_len_codons=max_len-2) for _ in range(N_VAL)]
    val_corpora[max_len] = torch.tensor([tok.encode(s, max_len) for s in seqs])
    print(f"  -> tensor {tuple(val_corpora[max_len].shape)}")

In [ ]:
@torch.no_grad()
def eval_mlm(model, token_ids, tokenizer, device, batch_size=64):
    """Calcula perplejidad MLM media sobre un corpus."""
    model.eval()
    total_loss, total_n = 0.0, 0
    for i in range(0, token_ids.size(0), batch_size):
        batch = token_ids[i:i+batch_size]
        inp, lbl = mlm_mask(batch, tokenizer)
        inp, lbl = inp.to(device), lbl.to(device)
        logits = model(inp)
        n = (lbl != -100).sum().item()
        if n == 0:
            continue
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), lbl.view(-1),
                                ignore_index=-100, reduction="sum")
        total_loss += loss.item()
        total_n += n
    return math.exp(total_loss / max(1, total_n))


def train_one_model(pos_mode, train_token_ids, tokenizer, epochs=4,
                    batch_size=64, lr=5e-4, device=device):
    torch.manual_seed(SEED)  # mismo init para los tres modelos
    model = FlexibleMLM(
        vocab_size=tokenizer.vocab_size, d_model=128, num_heads=4,
        num_layers=2, d_ff=512, max_len=512, dropout=0.1,
        pad_id=tokenizer.pad_id, pos_mode=pos_mode,
    ).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n=== Entrenando pos_mode={pos_mode}  |  {n_params:,} params ===")
    for epoch in range(epochs):
        perm = torch.randperm(train_token_ids.size(0))
        model.train()
        epoch_loss, epoch_n = 0.0, 0
        for i in range(0, train_token_ids.size(0), batch_size):
            idx = perm[i:i+batch_size]
            batch = train_token_ids[idx]
            inp, lbl = mlm_mask(batch, tokenizer)
            inp, lbl = inp.to(device), lbl.to(device)
            logits = model(inp)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), lbl.view(-1),
                                    ignore_index=-100)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item() * (lbl != -100).sum().item()
            epoch_n += (lbl != -100).sum().item()
        train_ppl = math.exp(epoch_loss / max(1, epoch_n))
        print(f"  Epoch {epoch+1}/{epochs}  train_ppl = {train_ppl:.2f}")
    return model


# Entrenar los tres modelos
models = {}
for mode in ["sinusoidal", "rope", "alibi"]:
    models[mode] = train_one_model(mode, train_token_ids, tok, epochs=4)

In [ ]:
# Evaluar en longitudes {64, 128, 256}
print("\n" + "=" * 70)
print(f"{'pos_mode':<14}{'len=64':<14}{'len=128':<14}{'len=256':<14}{'ratio 256/64'}")
print("=" * 70)

results = {}
for mode, model in models.items():
    results[mode] = {}
    row = f"{mode:<14}"
    for L in [64, 128, 256]:
        ppl = eval_mlm(model, val_corpora[L], tok, device)
        results[mode][L] = ppl
        row += f"{ppl:<14.2f}"
    ratio = results[mode][256] / results[mode][64]
    row += f"{ratio:.2f}x"
    print(row)
print("=" * 70)
print("\nRecordatorio: longitud de entrenamiento = 64 codones.")
print("Cuanto MAS BAJA la perplejidad y MAS CERCA de 1.0x el ratio, mejor extrapolacion.")

In [ ]:
# Visualizacion
fig, ax = plt.subplots(figsize=(9, 5.5))
markers = {"sinusoidal": "o", "rope": "s", "alibi": "^"}
colors = {"sinusoidal": "#d62728", "rope": "#1f77b4", "alibi": "#2ca02c"}

for mode in ["sinusoidal", "rope", "alibi"]:
    Ls = sorted(results[mode].keys())
    ppls = [results[mode][L] for L in Ls]
    ax.plot(Ls, ppls, marker=markers[mode], color=colors[mode],
            label=mode, markersize=11, linewidth=2)

ax.axvline(TRAIN_MAX_LEN, color="gray", linestyle="--", alpha=0.6,
           label=f"longitud de entrenamiento ({TRAIN_MAX_LEN})")
ax.axhline(tok.vocab_size, color="black", linestyle=":", alpha=0.4,
           label=f"baseline uniforme ({tok.vocab_size})")
ax.set_xlabel("Longitud de secuencia en evaluacion (codones)", fontsize=12)
ax.set_ylabel("Perplejidad MLM (validacion)", fontsize=12)
ax.set_title("Extrapolacion de codificaciones posicionales\nentrenado a longitud 64, evaluado a 64/128/256",
             fontsize=13)
ax.set_xscale("log", base=2)
ax.set_xticks([64, 128, 256])
ax.set_xticklabels(["64", "128", "256"])
ax.legend(loc="best", fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Discusión — por qué la literatura del proyecto eligió RoPE/ALiBi

Lo que acabas de observar empíricamente es exactamente lo que motivó las decisiones de diseño de los modelos que vas a usar en el Módulo 4:

### Nucleotide Transformer v2 y ESM-2 → RoPE

> *"NT-v2 uses [...] 6mer tokenization on DNA sequences. This coarser tokenization allows the model to accept sequences as long as 12kbp, which is particularly important for capturing long range genomic interactions. As described in the Nucleotide Transformer paper, the second version of the model, similar to ESM, makes use of the RoPE embeddings scheme."* — Boshar et al. 2024

ESM-2 pasó de embeddings posicionales aprendidos (ESM-1b) a RoPE precisamente para extrapolación. Para conotoxinas — donde quieres procesar precursores completos en una pasada (~600 nt) usando NT-v2 con contexto 12 kb — RoPE no es opcional, es lo que hace viable el problema.

### DNABERT-2 → ALiBi

> *"DNABERT2 also replaces the learned positional embeddings of its predecessor with Attention with Linear Biases (ALiBi), a method which, similar to RoPE, removes the constraint on input length."* — Boshar et al. 2024

DNABERT (v1) tenía el famoso límite duro de 512 tokens con embeddings aprendidos. DNABERT-2 con ALiBi puede procesar secuencias mucho más largas sin reentrenamiento.

### Cuándo usar cuál

| Criterio | RoPE | ALiBi |
|---|---|---|
| **Calidad de extrapolación** | Buena (hasta ~2x longitud entrenamiento) | Excelente (incluso 10-100x según Press et al.) |
| **Parámetros añadidos** | 0 (sólo cos/sin caches) | 0 (sólo pendientes deterministas) |
| **Implementación** | Modifica Q y K | Añade bias a scores |
| **Compatibilidad con FlashAttention-2** | Nativa (kernel optimizado) | Soporte estándar |
| **Adoptado por** | ESM-2, NT-v2, LLaMA, Mistral, casi todo LLM moderno | DNABERT-2, BLOOM |

La tendencia 2024-2025 favorece **RoPE** en el mainstream LLM, pero ALiBi mantiene ventaja en regímenes de extrapolación extrema. Para tu proyecto, ambos son superiores a sinusoidal y a embeddings aprendidos.

## 7. Ejercicios

### 7.1 — Embeddings posicionales aprendidos (estilo DNABERT-1)

Añade un cuarto modo `pos_mode="learned"` que use `nn.Embedding(max_len, d_model)`. Entrénalo en condiciones idénticas y compáralo con los otros tres. Predicción: degradará peor que sinusoidal en len=128 y catastróficamente en len=256.

### 7.2 — Variar el `base` de RoPE

El parámetro `base = 10000.0` en `precompute_rope_cache` controla la longitud de onda de las rotaciones. Modelos como LLaMA-2 cambiaron a `base=500000` para soportar contextos más largos. Reentrena con `base ∈ {1000, 10000, 100000}` y compara extrapolación a len=512.

### 7.3 — Implementar ALiBi causal

Implementa la versión causal original del paper (sólo permite atender hacia atrás). Pista: usa `triu_(diagonal=1)` con `fill_value=-inf` antes del softmax.

### 7.4 — Pendientes ALiBi aprendibles

Press et al. dejan las pendientes m_h fijas, pero algunos trabajos posteriores las hacen aprendibles. Modifica `FlexibleMHA` para hacer `slopes` un `nn.Parameter` con `softplus` para garantizar positividad. Mejora o no?

### 7.5 — RoPE con NTK-aware scaling

Cuando se quiere extender un modelo RoPE pre-entrenado a contextos mayores, la técnica *NTK-aware interpolation* reescala las frecuencias en lugar de las posiciones. Búscala (blog post de bloc97 en r/LocalLLaMA, julio 2023) e impleméntala como variante del cache de RoPE.

## 8. Conclusión y conexión con el resto del bootcamp

✓ Has implementado **RoPE** y **ALiBi** desde cero, sin librerías externas.
✓ Has demostrado empíricamente la propiedad de invarianza relativa de RoPE y el sesgo de localidad de ALiBi.
✓ Has validado el hallazgo central de Press et al. (2021) y Su et al. (2021): los esquemas posicionales modernos extrapolan mejor que la suma sinusoidal del paper original.

### Lo que viene en Módulos posteriores

| Módulo | Lo que sabes ahora aplicado allí |
|---|---|
| **Módulo 3** (pre-entrenamiento escalado) | Al pre-entrenar sobre genomas Conidae multiespecie a contexto ~12 kb, RoPE/ALiBi son las únicas opciones viables |
| **Módulo 4** (PEFT con LoRA) | Cuando cargues `InstaDeepAI/nucleotide-transformer-v2-50m-multi-species` (RoPE) o `zhihan1996/DNABERT-2-117M` (ALiBi) desde HuggingFace, los entenderás por dentro |
| **Módulo 5** (interpretabilidad) | Las visualizaciones de atención requieren saber cómo se inyecta la posición |
| **Módulo 6** (ConoGen generativo) | Modelos autorregresivos tipo ProGen usan RoPE — el muestreo a longitudes mayores que entrenamiento requiere esta arquitectura |

### Bibliografía consultada en este notebook

* **Su, J., Lu, Y., Pan, S., Wen, B., & Liu, Y. (2021).** RoFormer: Enhanced Transformer with Rotary Position Embedding. *arXiv:2104.09864*. (RoPE original)
* **Press, O., Smith, N. A., & Lewis, M. (2022).** Train Short, Test Long: Attention with Linear Biases Enables Input Length Extrapolation. *ICLR*. (ALiBi)
* **Lin, Z. et al. (2023).** Evolutionary-scale prediction of atomic-level protein structure with a language model. *Science 379:1123-1130*. (ESM-2, usa RoPE)
* **Dalla-Torre, H. et al. (2025).** Nucleotide Transformer: building and evaluating robust foundation models for human genomics. *Nature Methods 22:287-297*. (NT-v2, usa RoPE)
* **Zhou, Z. et al. (2024).** DNABERT-2: Efficient Foundation Model and Benchmark for Multi-Species Genomes. *ICLR*. (usa ALiBi)

In [ ]:
# Resumen de resultados finales
print("=" * 70)
print("RESUMEN DEL EXPERIMENTO DE EXTRAPOLACION")
print("=" * 70)
print(f"Entrenamiento: longitud {TRAIN_MAX_LEN} codones, {N_TRAIN} secuencias, 4 epocas\n")
for mode in ["sinusoidal", "rope", "alibi"]:
    print(f"  {mode:<11} -> ppl@64={results[mode][64]:.2f}  "
          f"ppl@128={results[mode][128]:.2f}  ppl@256={results[mode][256]:.2f}")
print("\nModulo 1 completo. Listo para Modulo 2 (tokenizacion biologica).")